PURPOSE :

To explain how quantum key distribution (BB84) works step by step.

## Introduction

Quantum Key Distribution (QKD) allows two users to generate a shared secret key.
The BB84 protocol uses quantum states so that any eavesdropping attempt
automatically introduces errors and can be detected.


In [1]:
import random
from qiskit import QuantumCircuit
from qiskit_aer import Aer


## Step 1: Alice Generates Random Bits and Bases

Alice creates random bits (0 or 1) and randomly chooses a basis
(`+` or `x`) to encode each bit.


In [2]:
n = 10
alice_bits = [random.randint(0, 1) for _ in range(n)]
alice_bases = [random.choice(['+', 'x']) for _ in range(n)]

alice_bits, alice_bases


([0, 0, 1, 0, 1, 1, 1, 1, 1, 0],
 ['+', 'x', '+', 'x', '+', '+', 'x', '+', 'x', '+'])

## Step 2: Quantum Encoding

Each bit is encoded into a quantum state.
If the basis is `x`, a Hadamard gate is applied.


In [3]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)
qc


## Security of BB84

If an eavesdropper tries to measure the quantum states,
the states collapse and errors are introduced.
By checking the error rate, Alice and Bob can detect intrusion.



## Note: realistic intercept-resend in `src/quantum_key_generator.py`

The simulation above shows a single qubit for illustration. The actual
implementation in `src/quantum_key_generator.py` models Eve more
carefully: she measures each qubit in a randomly chosen basis **and then
re-prepares a fresh qubit from her own measured bit and basis** before
forwarding it to Bob (a genuine intercept-resend attack), rather than
simply discarding the qubit. This produces the textbook ~25% QBER for
intercept-resend against BB84, which is what the intrusion-detection
threshold (11%) is calibrated against.

The sifted key that survives this exchange is also no longer discarded
after the error-rate check — it's run through HKDF-SHA256
(`src/key_derivation.py`) to derive the actual AES key used to encrypt
the message. See `CHANGELOG.md` for details.

## Conclusion

This notebook demonstrates the fundamental idea of BB84:
quantum properties make eavesdropping detectable,
providing secure key distribution.
